# 국내·해외 × 동물성·식물성·기준규격외·가공식품 양성률

- 데이터: `리스테리아ML/data/listeria_df_joined.parquet` (lv1) + `리스테리아ML/data/X.parquet` (`src_route`)
- 국내·해외: `X.parquet` 의 `src_route` — 로마자로 적힌 한국 주소 78행을 overseas → domestic 으로 옮긴 값 (`EXP/1/1_데이터정제.ipynb`)
- 분류: 품목 계층 `lv1_nm`. 넷에 들지 않는 lv1(및 코드 없는 행)은 '그 외'로 따로 보인다
- **양성률 = 칸의 검출 수 ÷ 전체 검출 수 (%)**

In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)
DATA = r"C:\cheon\cheon_wokespace\listeria2026\리스테리아ML\data"
d = pd.read_parquet(f"{DATA}/listeria_df_joined.parquet",
                    columns=["lims_data_key", "train_prdlst_code", "train_prdlst_nm", "lv1_cd", "lv1_nm", "judge_result", "date_basis"])
d["y"] = (d["judge_result"] == "검출").astype(int)
N_POS = int(d["y"].sum())
print(f"전체 {len(d):,}행 · 전체 양성 {N_POS:,}건")
x = pd.read_parquet(f"{DATA}/X.parquet", columns=["lims_data_key", "src_route"])
d = d.merge(x.rename(columns={"src_route": "src_route_fixed"}), on="lims_data_key", how="left", validate="one_to_one")
assert d["src_route_fixed"].notna().all()
print(d["src_route_fixed"].value_counts())

전체 676,537행 · 전체 양성 1,969건


src_route_fixed
domestic    628973
overseas     47564
Name: count, dtype: int64


In [2]:
print(d["lv1_nm"].astype("string").fillna("(코드 없음)").value_counts())

lv1_nm
가공식품       333276
기준규격외      320550
동물성         10204
식품접객업소       9654
식물성          2098
건강기능식품        446
(코드 없음)       170
첨가물           139
Name: count, dtype: int64[pyarrow]


## 양성률 표 (분모 = 전체 양성)

In [3]:
CATS = ["동물성", "식물성", "기준규격외", "가공식품"]
d["분류"] = d["lv1_nm"].astype("string").where(d["lv1_nm"].isin(CATS), "그 외")
d["국내외"] = d["src_route_fixed"].astype("string").map({"domestic": "국내", "overseas": "해외"})
pos = pd.crosstab(d["국내외"], d["분류"], values=d["y"], aggfunc="sum", margins=True, margins_name="합계").fillna(0).astype(int)
order = [c for c in CATS + ["그 외"] if c in pos.columns] + ["합계"]
rate = (pos[order] / N_POS * 100).round(2)
print("양성률(%, 분모 = 전체 양성", N_POS, "건)")
rate

양성률(%, 분모 = 전체 양성 1969 건)


분류,동물성,식물성,기준규격외,가공식품,그 외,합계
국내외,,,,,,
국내,0.61,0.76,72.68,19.6,0.0,93.65
해외,3.15,0.00,0.00,3.2,0.0,6.35
합계,3.76,0.76,72.68,22.8,0.0,100.00


## 참고: 칸별 검출 수 · 행 수 · 칸 안 검출률

In [4]:
rows = pd.crosstab(d["국내외"], d["분류"], margins=True, margins_name="합계")[order]
print("검출 수"); display(pos[order])
print("행 수"); display(rows)
print("칸 안 검출률(%, 분모 = 칸의 행 수)"); display((pos[order] / rows * 100).round(4))

검출 수


분류,동물성,식물성,기준규격외,가공식품,그 외,합계
국내외,,,,,,
국내,12,15,1431,386,0,1844
해외,62,0,0,63,0,125
합계,74,15,1431,449,0,1969


행 수


분류,동물성,식물성,기준규격외,가공식품,그 외,합계
국내외,,,,,,
국내,1445,2098,320536,294489,10405,628973
해외,8759,0,14,38787,4,47564
합계,10204,2098,320550,333276,10409,676537


칸 안 검출률(%, 분모 = 칸의 행 수)


분류,동물성,식물성,기준규격외,가공식품,그 외,합계
국내외,,,,,,
국내,0.8304,0.715,0.4464,0.1311,0.0,0.2932
해외,0.7078,NaN,0.0000,0.1624,0.0,0.2628
합계,0.7252,0.715,0.4464,0.1347,0.0,0.2910
